[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EstebanGZam/taller_agente/blob/main/secop_agent_colab.ipynb)

# Workshop 1 · SECOP procurement analyst agent

This notebook reproduces the complete Workshop 1 deliverable:

1. Clone the repository and install the package.
2. Load the keys from Colab secrets.
3. Offline checks (no language model, no cost).
4. One sample question to the Q&A agent.
5. Experiments: configuration A (baseline) vs. B (improved) in LangSmith, plus the comparison table.
6. Eligibility of the company in the 20 tender documents.
7. Web frontend (Gradio).
8. Where to look in LangSmith.

Workshop criterion: **at least 85% accuracy at the lowest possible cost**. The agents answer in Spanish
because the reference answers are in Spanish.

## 1. Clone the repository and install the package

In [1]:
import os

REPO_URL = "https://github.com/EstebanGZam/taller_agente"
# On Colab the repository is cloned into /content; elsewhere, run the notebook from the repository root.
REPO_DIR = "/content/taller_agente" if os.path.isdir("/content") else os.getcwd()
if not os.path.exists(REPO_DIR):
    !git clone -q -b main {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull -q
!pip install -q -e ".[dev]"  # the secop_agent package plus pytest and ruff for the offline checks

/home/esteban/Documents/University/Tenth semester/IA/unit-iii/taller_agente



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


## 2. Keys and configuration

Store `ANTHROPIC_API_KEY` and `LANGSMITH_API_KEY` in Colab **Secrets** (key icon in the left sidebar) and
grant this notebook access. Missing keys are requested with `getpass`. Keys are never printed.

Do not set `LANGSMITH_AGENT_ID` or `LANGSMITH_AGENT_ENVIRONMENT`: together with `LANGSMITH_PROJECT` they
stop LangSmith from recording traces, so they are removed here.

In [2]:
import os
from getpass import getpass

for name in ("LANGSMITH_AGENT_ID", "LANGSMITH_AGENT_ENVIRONMENT"):
    os.environ.pop(name, None)


def read_secret(name: str) -> str:
    """Colab secret, or an interactive prompt when the secret is missing or not shared."""
    try:
        from google.colab import userdata

        value = userdata.get(name)
        if value:
            return value
    except Exception:  # Outside Colab, or the secret does not exist / has no access.
        pass
    return getpass(f"{name}: ")


for key in ("ANTHROPIC_API_KEY", "LANGSMITH_API_KEY"):
    if not os.environ.get(key):
        os.environ[key] = read_secret(key)

TEAM_NAME = "estebangzam"  # <- change to your team name
os.environ.update({
    "TEAM_NAME": TEAM_NAME,
    "LANGSMITH_TRACING": "true",
    "LANGSMITH_PROJECT": f"secop-taller1-{TEAM_NAME}",
    "MODEL": "anthropic:claude-haiku-5-5",
    "DATA_MODE": "sample",
    "AGENT_CONFIG": "B",
})
print("Keys loaded:", {key: bool(os.environ.get(key)) for key in ("ANTHROPIC_API_KEY", "LANGSMITH_API_KEY")})
print("LangSmith project:", os.environ["LANGSMITH_PROJECT"])

Keys loaded: {'ANTHROPIC_API_KEY': True, 'LANGSMITH_API_KEY': True}
LangSmith project: secop-taller1-estebangzam


## 3. Offline checks

`secop_agent.procurement_data` runs the data tools in plain Python. `pytest` checks, without keys or
network, that the tools return the facts behind every reference question, that configuration A is the
course's original agent, and the 20 eligibility verdicts; `ruff` checks the code style.

In [3]:
!python -m secop_agent.procurement_data
!python -m pytest -q && ruff check .

Financial indicators: {'liquidity_ratio': 1.7, 'debt_ratio_pct': 62.0, 'interest_coverage_ratio': 2.5, 'working_capital': 182000000, 'return_on_equity': 0.105, 'return_on_assets': 0.04, 'financial_statements_date': '2025-12-31'}
Search modality='licitaci' (sample mode): {'count': 2, 'total_budget': 54120784298.0, 'average_budget': 27060392149.0, 'zero_budget_processes': 0}


.....

.......................................

.....

.....

...

.............

.................................................

... [ 79%]
.....................................                                    [100%]


=============================== warnings summary ===============================
tests/test_eligibility_agent.py::test_root_trace_metadata_offline
  /home/esteban/Documents/University/Tenth semester/IA/unit-iii/.venv-taller/lib/python3.13/site-packages/langsmith/client.py:726: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
    warnings.warn(

tests/test_eligibility_agent.py::test_app_import_and_build_without_keys
  /home/esteban/Documents/University/Tenth semester/IA/unit-iii/taller_agente/secop_agent/web_app.py:188: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
    with gr.Blocks(

tests/test_eligibility_agent.py::test_app_import_and_build_without_keys
  /home/esteban/Documents/University/Tenth semester/IA/unit-iii/taller_agente/secop_agent/web_app.py:188: DeprecationWarning: The 'css' parameter in the Blocks constructor will be remov

All checks passed!


## 4. One question to the agent

`ask()` returns the answer and the tools the agent decided to call. `CostTracker` adds up tokens and cost
with Anthropic's public rates.

In [4]:
from secop_agent.costs import CostTracker
from secop_agent.qa_agent import ask, normalize_config
from secop_agent.settings import model_from_env

tracker = CostTracker(model_from_env())
result = ask("¿Nuestro capital de trabajo alcanza el 40 % del presupuesto del proceso "
             "FLO-INFRA-SA-MC-007-2025?", callbacks=[tracker])
print(result.answer)
print("\nTools used:", ", ".join(result.tools_used) or "none")
print(f"Configuration {normalize_config()} · {model_from_env()} ·", tracker.summary())

Sí. Nuestro capital de trabajo es de $182.000.000, que equivale al 59,5 % del presupuesto del proceso FLO-INFRA-SA-MC-007-2025 (Alcaldía de Floridablanca, Selección Abreviada de Menor Cuantía, presupuesto oficial de $305.813.941). El 40 % exigido son $122.325.576, así que cumplimos el requisito con holgura.

Tools used: search_processes, compute_financial_indicators
Configuration B · anthropic:claude-haiku-5-5 · {'model_calls': 3, 'input_tokens': 8985, 'output_tokens': 327, 'cache_read_tokens': 0, 'cost_usd': 0.001062, 'cost_complete': True}


## 5. Experiments: configuration A vs. B

- **A**: the course's original prompt and tool descriptions (baseline).
- **B**: precise tool descriptions, compact output with a summary and data-quality warnings, and a prompt
  with one rule per question type.

Each run evaluates the 18 reference questions in LangSmith (dataset `secop-taller1-referencia`) with two
evaluators (tool usage and answer correctness by an LLM judge) and saves
`results/qa_experiment_<prefix>.json` with accuracy, tokens, cost and latency.

In [5]:
!AGENT_CONFIG=A python -m secop_agent.evaluate_qa A-baseline
!AGENT_CONFIG=B python -m secop_agent.evaluate_qa B-improved

View the evaluation results for experiment: 'A-baseline-f20d7ff7' at:
https://smith.langchain.com/o/be862a28-1dbf-4dec-b6c4-f58119a8f835/datasets/eadd73f4-6170-4e3d-9565-08affe410e31/compare?selectedSessions=eee715ad-faa1-41bf-ac27-a380ce2d5193


0it [00:00, ?it/s]

1it [00:05,  5.19s/it]

2it [00:07,  3.29s/it]

3it [00:09,  2.81s/it]

4it [00:10,  2.24s/it]

...

14it [00:23,  1.42s/it]

16it [00:25,  1.17s/it]

17it [00:26,  1.26s/it]

18it [00:28,  1.57s/it]



Experiment B-improved-4e8aed68 · AGENT_CONFIG=B · anthropic:claude-haiku-5-5
  Tool usage:            100.0 %
  Correct answer:        100.0 %
  Both correct:          100.0 %
  Input / output tokens: 97,358 / 5,319
  Total cost (ours):     USD 0.012395 · per question USD 0.000689
  Total cost (LangSmith): USD 0.012780
  Latency p50 / p95:     3.01 s / 3.69 s

Saved to /home/esteban/Documents/University/Tenth semester/IA/unit-iii/taller_agente/results/qa_experiment_B-improved.json
Review the experiment in LangSmith > Datasets & Experiments.


In [6]:
from pathlib import Path

from IPython.display import Markdown, display

!python -m secop_agent.compare_experiments A-baseline B-improved > /dev/null
display(Markdown(Path("results/qa_comparison.md").read_text(encoding="utf-8")))

# Q&A agent configurations · Workshop 1

Reference set (18 questions, `DATA_MODE=sample`). Our cost comes from `secop_agent.costs` (Anthropic's public rates); LangSmith may not know the model's rate.

| | A-baseline | B-improved |
|---|---|---|
| LangSmith experiment | `A-baseline-f20d7ff7` | `B-improved-4e8aed68` |
| Configuration | A | B |
| Model | `anthropic:claude-haiku-5-5` | `anthropic:claude-haiku-5-5` |
| Change | Course's original prompt and tool descriptions (baseline) | Precise descriptions, compact output with summary and warnings, reference filter, exact modality and sorting; prompt with one rule per question type |
| Questions | 18 | 18 |
| Tool usage | 100.0 % | 100.0 % |
| Correct answer | 83.3 % | 100.0 % |
| Both correct | 83.3 % | 100.0 % |
| Input / output tokens | 77,684 / 8,968 | 97,358 / 5,319 |
| Total cost (ours) | USD 0.012252 | USD 0.012395 |
| Cost per question | USD 0.000681 | USD 0.000689 |
| Cost per correct answer | USD 0.000817 | USD 0.000689 |
| Total cost (LangSmith) | USD 0.012154 | USD 0.012780 |
| Latency p50 | 4.05 s | 3.01 s |
| Latency p95 | 6.32 s | 3.69 s |

**Verdict:** Meeting the 85% correct-answer criterion: B-improved (100.0%). Cheapest per correct answer: B-improved (USD 0.000689). Recommended: B-improved (>= 85% at the lowest cost per question).


## 6. Eligibility in the 20 tender documents

Reads the 20 PDFs, extracts the requirements, checks them against `empresa_multiproposito.json` and
compares each verdict (ELIGIBLE, NOT_ELIGIBLE, NOT_APPLICABLE) with the key in `data/eligibility_key.json`.
Name, reference and cost of every process are recorded in `results/eligibility_processes.{csv,md,json}`.

In [7]:
from pathlib import Path

from IPython.display import Markdown, display

!python -m secop_agent.batch_eligibility --company empresa_multiproposito.json
display(Markdown(Path("results/eligibility_processes.md").read_text(encoding="utf-8")))

TEAM_NAME: estebangzam
Repository: https://github.com/EstebanGZam/taller_agente
Processes: 20 · 20/20 · accuracy 100.00%
Verdicts: NOT_APPLICABLE: 12, ELIGIBLE: 6, NOT_ELIGIBLE: 2
Total cost USD: 0.044076
Mean cost per process USD: 0.002204
Cost complete: True
p50 latency: 13.108 s
Tokens: input 219938, output 44165
Model: anthropic:claude-haiku-5-5
Strict SMMLV: False
Reference key uses nominal pesos.


# Tender eligibility

TEAM_NAME: estebangzam
Repository: https://github.com/EstebanGZam/taller_agente
Processes: 20 · 20/20 · accuracy 100.00%
Verdicts: NOT_APPLICABLE: 12, ELIGIBLE: 6, NOT_ELIGIBLE: 2
Total cost USD: 0.044076
Mean cost per process USD: 0.002204
Cost complete: True
p50 latency: 13.108 s
Tokens: input 219938, output 44165
Model: anthropic:claude-haiku-5-5
Strict SMMLV: False
Reference key uses nominal pesos.

| n | file | reference | entity | modality | official_budget | verdict | expected_verdict | correct | summary | tools_used | model_calls | input_tokens | output_tokens | cost_usd | latency_s | model |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | 01_EDP_545_2022.pdf | EDP-545-2022 | DEPARTAMENTO ADMINISTRATIVO NACIONAL DE ESTADISTICA (DANE) | Contratación directa | 57333333.0 | NOT_APPLICABLE | NOT_APPLICABLE | True | Direct hiring without a competitive call for bids. | get_company_profile, extract_requirements, verify_requirements | 4 | 11287 | 1720 | 0.0019887 | 12.39 | anthropic:claude-haiku-5-5 |
| 2 | 02_CD_55_2021_3516.pdf | CD-55-2021-3516 | DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA | Contratación directa | 2917200000.0 | NOT_APPLICABLE | NOT_APPLICABLE | True | Direct hiring without a competitive call for bids. | extract_requirements, get_company_profile, verify_requirements | 4 | 11538 | 2041 | 0.0021743 | 19.677 | anthropic:claude-haiku-5-5 |
| 3 | 03_RE_CGBM_049_2024.pdf | RE-CGBM-049-2024 | COLEGIO GABRIEL BETANCOURT MEJIA I.E.D | Contratación régimen especial (con ofertas) | 5500000.0 | ELIGIBLE | ELIGIBLE | True | Meets all verified requirements. | extract_requirements, verify_requirements | 5 | 10058 | 2453 | 0.0022323 | 14.364 | anthropic:claude-haiku-5-5 |
| 4 | 04_CD_CPS_0720_2026.pdf | CD-CPS-0720-2026 | INSTITUTO NACIONAL DE VÍAS (INVIAS) | Contratación directa | 48620250.0 | NOT_APPLICABLE | NOT_APPLICABLE | True | Direct hiring without a competitive call for bids. | get_company_profile, extract_requirements, verify_requirements | 4 | 11339 | 1600 | 0.0019339 | 12.243 | anthropic:claude-haiku-5-5 |
| 5 | 05_PS_147_2025.pdf | PS-147-2025 | MUNICIPIO DE SAN MIGUEL PUTUMAYO | Contratación directa | 10000000.0 | NOT_APPLICABLE | NOT_APPLICABLE | True | Direct hiring without a competitive call for bids. | extract_requirements, verify_requirements | 4 | 8380 | 1794 | 0.001735 | 11.338 | anthropic:claude-haiku-5-5 |
| 6 | 06_SINF_CPS_050_23.pdf | SINF-CPS-050-23 | MUNICIPIO DE PIEDECUESTA | Contratación directa | 12000000.0 | NOT_APPLICABLE | NOT_APPLICABLE | True | Direct hiring without a competitive call for bids. | extract_requirements, get_company_profile, verify_requirements | 4 | 11659 | 1957 | 0.0021444 | 12.611 | anthropic:claude-haiku-5-5 |
| 7 | 07_135116.pdf | 135116 | CORPORACIÓN COLOMBIANA DE INVESTIGACIÓN AGROPECUARIA (AGROSAVIA) | Contratación régimen especial | 0.0 | ELIGIBLE | ELIGIBLE | True | Meets all verified requirements. | get_company_profile, extract_requirements, verify_requirements | 5 | 13933 | 3304 | 0.0030453 | 18.744 | anthropic:claude-haiku-5-5 |
| 8 | 08_OC10_2494_2025.pdf | OC10-2494-2025 | EMPRESA SOCIAL DEL ESTADO BELLOSALUD | Contratación régimen especial | 4260200.0 | ELIGIBLE | ELIGIBLE | True | Meets all verified requirements. | extract_requirements, verify_requirements | 5 | 10176 | 2823 | 0.0024291 | 15.796 | anthropic:claude-haiku-5-5 |
| 9 | 09_SDIS_CPS_302405_2024.pdf | SDIS-CPS-302405-2024 | SECRETARÍA DISTRITAL DE INTEGRACIÓN SOCIAL | Contratación directa | 53579880.0 | NOT_APPLICABLE | NOT_APPLICABLE | True | Direct hiring without a competitive call for bids. | extract_requirements, get_company_profile, verify_requirements | 4 | 11146 | 1762 | 0.0019956 | 13.219 | anthropic:claude-haiku-5-5 |
| 10 | 10_009_2021.pdf | 009-2021 | ESAP TERRITORIAL BOYACÁ CASANARE | Contratación directa | 9108000.0 | NOT_APPLICABLE | NOT_APPLICABLE | True | Direct hiring without a competitive call for bids. | get_company_profile, extract_requirements, verify_requirements | 4 | 11389 | 1793 | 0.0020354 | 12.628 | anthropic:claude-haiku-5-5 |
| 11 | 11_FDLS_LP_098_2019.pdf | FDLS-LP-098-2019 | ALCALDÍA LOCAL DE SUMAPAZ | Licitación pública Obra Pública | 600000000.0 | ELIGIBLE | ELIGIBLE | True | Meets all verified requirements. | extract_requirements, get_company_profile, compute_financial_indicators, verify_requirements | 4 | 13004 | 2558 | 0.0025794 | 12.748 | anthropic:claude-haiku-5-5 |
| 12 | 12_FNV_001_2021.pdf | FNV-001-2021 | FONDO NACIONAL DE VIVIENDA (FONVIVIENDA) | Contratación directa | 448000000.0 | NOT_APPLICABLE | NOT_APPLICABLE | True | Direct hiring without a competitive call for bids. | extract_requirements, verify_requirements | 4 | 7795 | 1575 | 0.001567 | 9.746 | anthropic:claude-haiku-5-5 |
| 13 | 13_LP_DT_CAU_003_2019.pdf | LP-DT-CAU-003-2019 | INSTITUTO NACIONAL DE VÍAS (INVIAS) - DIRECCIÓN TERRITORIAL CAUCA | Licitación pública Obra Pública | 53520784298.0 | NOT_ELIGIBLE | NOT_ELIGIBLE | True | Does not meet: Working capital, Combined experience, Single-contract experience. | extract_requirements, get_company_profile, verify_requirements, compute_financial_indicators | 4 | 13358 | 2730 | 0.0027008 | 14.12 | anthropic:claude-haiku-5-5 |
| 14 | 14_FLO_INFRA_SA_MC_007_2025.pdf | FLO-INFRA-SA-MC-007-2025 | ALCALDÍA DE FLORIDABLANCA | Selección abreviada de menor cuantía | 305813941.0 | ELIGIBLE | ELIGIBLE | True | Meets all verified requirements. | extract_requirements, get_company_profile, compute_financial_indicators, verify_requirements | 4 | 13674 | 2898 | 0.0028164 | 14.807 | anthropic:claude-haiku-5-5 |
| 15 | 15_SA_004_2023.pdf | SA-004-2023 | INSTITUTO DE DEPORTES Y RECREACIÓN DE MEDELLÍN (INDER) | Selección abreviada subasta inversa | 270101513.0 | ELIGIBLE | ELIGIBLE | True | Meets all verified requirements. | extract_requirements, verify_requirements | 4 | 10337 | 2606 | 0.0023367 | 13.735 | anthropic:claude-haiku-5-5 |
| 16 | 16_CD_PS_GDCH_186SS_2024.pdf | CD-PS-GDCH-186SS-2024 | GOBERNACIÓN DEL CHOCÓ | Contratación directa | 24000000.0 | NOT_APPLICABLE | NOT_APPLICABLE | True | Direct hiring without a competitive call for bids. | extract_requirements, verify_requirements | 4 | 8671 | 1909 | 0.0018216 | 11.041 | anthropic:claude-haiku-5-5 |
| 17 | 17_5308_SASP01_2023.pdf | 5308/SASP01/2023 | ALCALDÍA DE GIRARDOTA | Enajenación de bienes con subasta (Artículo 99 del Decreto 1510 de 2013) | 86000000.0 | NOT_APPLICABLE | NOT_APPLICABLE | True | The entity sells assets; the company would be a buyer. | extract_requirements, verify_requirements | 4 | 8670 | 1885 | 0.0018095 | 12.998 | anthropic:claude-haiku-5-5 |
| 18 | 18_0511_2024.pdf | 0511-2024 | ESE CARMEN EMILIA OSPINA | Contratación régimen especial | 320000000.0 | NOT_ELIGIBLE | NOT_ELIGIBLE | True | Does not meet: Experience. | extract_requirements, get_company_profile, verify_requirements | 5 | 13404 | 3008 | 0.0028444 | 16.007 | anthropic:claude-haiku-5-5 |
| 19 | 19_CSS_297_2024.pdf | CSS-297-2024 | CXAYUCE JXUT E.S.E. | Contratación régimen especial | 21280000.0 | NOT_APPLICABLE | NOT_APPLICABLE | True | Direct hiring without a competitive call for bids. | extract_requirements, verify_requirements | 4 | 8321 | 1681 | 0.0016726 | 11.169 | anthropic:claude-haiku-5-5 |
| 20 | 20_CPS_009_2026.pdf | CPS-009-2026 | INSTITUCIÓN EDUCATIVA LAS COLINAS | Contratación régimen especial | 5400000.0 | NOT_APPLICABLE | NOT_APPLICABLE | True | Direct hiring without a competitive call for bids. | extract_requirements, get_company_profile, verify_requirements | 4 | 11799 | 2068 | 0.0022139 | 13.712 | anthropic:claude-haiku-5-5 |


In [8]:
!python -m secop_agent.evaluate_eligibility eligibility-haiku

View the evaluation results for experiment: 'eligibility-haiku-16b096ee' at:
https://smith.langchain.com/o/be862a28-1dbf-4dec-b6c4-f58119a8f835/datasets/8b84e4ec-607d-4457-843e-852e1fe0f9a8/compare?selectedSessions=9e401281-37d7-446a-9b4b-62d0aa2de796


0it [00:00, ?it/s]

1it [00:13, 13.09s/it]

2it [00:15,  6.54s/it]

3it [00:24,  8.01s/it]

4it [00:27,  5.72s/it]

...

16it [01:23,  4.92s/it]

17it [01:25,  4.20s/it]

18it [01:32,  5.12s/it]

19it [01:34,  4.10s/it]

20it [01:34,  4.73s/it]


## 7. Frontend

Opens the Gradio interface with a temporary public link (`share=True`). Stop the cell to close it. The cell is tagged
`skip-execution` so headless runs (nbclient) leave it out; Colab runs it normally.

In [ ]:
from secop_agent.web_app import build_demo

build_demo().queue().launch(share=True, debug=True)

## 8. Where to look in LangSmith

At [smith.langchain.com](https://smith.langchain.com):

- **Projects → `secop-taller1-<team>`**: one trace per agent question (section 4 and the frontend), with the
  reason–act tree, the tools called, and the tokens and latency of each step.
- **Datasets & Experiments → `secop-taller1-referencia`**: the `A-baseline-…` and `B-improved-…`
  experiments. Each row shows the output (with `metrics`: tokens, cost and latency) and the feedback
  columns `tool_usage`, `correct_answer` and `cost_usd`.
- Select both experiments and click **Compare** to see answers, scores and the **Latency**, **Tokens** and
  **Cost** columns side by side. When LangSmith does not know the model's rate the Cost column is empty;
  the reference cost is ours (`cost_usd` column and `results/qa_comparison.md`).
- Each experiment's metadata holds `agent_config`, `model`, `judge_model`, `data_mode` and `team_name`;
  the traces carry the tags `config-A` / `config-B`.
- **Datasets & Experiments**: the `eligibility-haiku-…` experiment of the eligibility evaluation.